<a href="https://colab.research.google.com/github/Samarpan-techy/AGI-MEM/blob/main/miniprojectsam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# 🚀 AGI MEMORY PLATFORM (Gradio 6.x + Groq + SQLite)
# ==============================================================================

!pip install -q gradio groq

import gradio as gr
import sqlite3
import subprocess
import json
from datetime import datetime
from groq import Groq

DB_NAME = "colab_memory.db"

# --- DATABASE SETUP & HOOKS ---
def init_db():
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute('''CREATE TABLE IF NOT EXISTS sessions (
        id INTEGER PRIMARY KEY,
        name TEXT,
        last_heartbeat TIMESTAMP
    )''')
    c.execute('''CREATE TABLE IF NOT EXISTS messages (
        id INTEGER PRIMARY KEY,
        session_id INTEGER,
        role TEXT,
        content TEXT
    )''')
    c.execute("INSERT OR IGNORE INTO sessions (id, name, last_heartbeat) VALUES (1, 'agi-memory', CURRENT_TIMESTAMP)")
    conn.commit()
    conn.close()

def post_tool_use_hook(session_id, tool_name, log_list):
    """Fires after every tool call to update heartbeat and activity log."""
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("UPDATE sessions SET last_heartbeat = CURRENT_TIMESTAMP WHERE id = ?", (session_id,))
    conn.commit()
    conn.close()

    log_entry = f"[{datetime.now().strftime('%H:%M:%S')}] Hook: {tool_name} executed. Heartbeat updated."
    log_list.append(log_entry)

# --- TOOLS ---
def run_terminal(command):
    try:
        res = subprocess.run(command, shell=True, capture_output=True, text=True, timeout=15)
        return (res.stdout if res.stdout else res.stderr).strip() or "Command executed successfully."
    except Exception as e:
        return f"Error: {str(e)}"

def search_memory(query, session_id=1):
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("SELECT content FROM messages WHERE session_id = ? AND content LIKE ? ORDER BY id DESC LIMIT 3", (session_id, f"%{query}%"))
    rows = c.fetchall()
    conn.close()
    return "\n".join([r[0] for r in rows]) or "No memories found."

TOOLS_SCHEMA = [
    {
        "type": "function",
        "function": {
            "name": "run_terminal",
            "description": "Execute a shell command.",
            "parameters": {"type": "object", "properties": {"command": {"type": "string"}}, "required": ["command"]},
        },
    },
    {
        "type": "function",
        "function": {
            "name": "search_memory",
            "description": "Search past conversation memory.",
            "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        },
    }
]

# --- AGENT LOGIC ---
def agent_chat(user_msg, history, api_key, session_id, terminal_history, activity_logs):
    if not api_key:
        history = history or []
        history.append({"role": "user", "content": user_msg})
        history.append({"role": "assistant", "content": "⚠️ Please enter your Groq API Key."})
        return history, terminal_history, "\n".join(activity_logs or [])

    if history is None: history = []
    if activity_logs is None: activity_logs = []
    if not terminal_history: terminal_history = "anthony@colab:~$"

    client = Groq(api_key=api_key)
    # ✅ FIXED: Updated to the new supported Groq model
    model = "openai/gpt-oss-120b"

    # Save user message to DB
    conn = sqlite3.connect(DB_NAME)
    c = conn.cursor()
    c.execute("INSERT INTO messages (session_id, role, content) VALUES (?, ?, ?)", (session_id, "user", user_msg))
    conn.commit()
    conn.close()

    # Append new user message to Gradio history
    history.append({"role": "user", "content": user_msg})

    # Build messages for Groq API
    messages = [{"role": "system", "content": "You are an expert coding agent. Use tools when necessary. Be extremely brief."}]
    for msg in history:
        messages.append(msg)

    activity_logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] Agent processing...")

    try:
        # First LLM Call
        response = client.chat.completions.create(model=model, messages=messages, tools=TOOLS_SCHEMA, tool_choice="auto")
        response_msg = response.choices[0].message

        if response_msg.tool_calls:
            # Convert Groq object to dict for API message history
            messages.append(response_msg.model_dump(exclude_none=True))

            for tc in response_msg.tool_calls:
                args = json.loads(tc.function.arguments)

                if tc.function.name == "run_terminal":
                    cmd = args.get("command", "")
                    activity_logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] Tool: run_terminal('{cmd}')")
                    terminal_history += f"\n$ {cmd}\n"
                    result = run_terminal(cmd)
                    terminal_history += f"{result}\n"

                elif tc.function.name == "search_memory":
                    query = args.get("query", "")
                    activity_logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] Tool: search_memory('{query}')")
                    result = search_memory(query, session_id)

                # Fire Hook
                post_tool_use_hook(session_id, tc.function.name, activity_logs)
                messages.append({"role": "tool", "tool_call_id": tc.id, "content": result})

            # Second LLM Call (Final Answer after tools execute)
            final_res = client.chat.completions.create(model=model, messages=messages)
            final_content = final_res.choices[0].message.content
        else:
            final_content = response_msg.content

        # Save assistant response to DB
        conn = sqlite3.connect(DB_NAME)
        c = conn.cursor()
        c.execute("INSERT INTO messages (session_id, role, content) VALUES (?, ?, ?)", (session_id, "assistant", final_content))
        conn.commit()
        conn.close()

        activity_logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] Response generated.")

        # Append to Gradio history
        history.append({"role": "assistant", "content": final_content})

        return history, terminal_history, "\n".join(activity_logs)

    except Exception as e:
        activity_logs.append(f"[{datetime.now().strftime('%H:%M:%S')}] Error: {str(e)}")
        history.append({"role": "assistant", "content": f"❌ Error: {str(e)}"})
        return history, terminal_history, "\n".join(activity_logs)

# --- UI LAYOUT ---
init_db()

with gr.Blocks(title="AGI Memory - Colab Web") as demo:
    gr.Markdown("## 🧠 AGI Memory Platform (Groq Powered)")

    with gr.Row():
        # LEFT COLUMN (Sidebar)
        with gr.Column(scale=1, min_width=200):
            api_key_input = gr.Textbox(label="Groq API Key", type="password", placeholder="gsk_...")
            gr.Markdown("### Sessions")
            session_list = gr.Dataframe(
                value=[["agi-memory", "Active"]],
                headers=["Project", "Status"],
                interactive=False
            )

        # MIDDLE COLUMN (Chat)
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(label="Frontend", height=500)
            with gr.Row():
                msg_input = gr.Textbox(show_label=False, placeholder="Ask anything...", scale=4)
                send_btn = gr.Button("Send", variant="primary", scale=1)

        # RIGHT COLUMN (Activity Log)
        with gr.Column(scale=1, min_width=200):
            gr.Markdown("### Activity Timeline")
            activity_output = gr.Textbox(
                label="Hooks & Tools",
                lines=20,
                max_lines=20,
                interactive=False,
                value="System ready..."
            )

    # BOTTOM ROW (Terminal)
    with gr.Row():
        terminal_output = gr.Textbox(
            label="Terminal Output",
            lines=8,
            max_lines=8,
            interactive=False,
            value="anthony@colab:~$"
        )

    # State variables
    session_state = gr.State(1)
    terminal_state = gr.State("anthony@colab:~$")
    activity_state = gr.State(["System initialized."])

    # Event Listeners
    send_btn.click(
        fn=agent_chat,
        inputs=[msg_input, chatbot, api_key_input, session_state, terminal_state, activity_state],
        outputs=[chatbot, terminal_output, activity_output]
    ).then(fn=lambda: "", inputs=None, outputs=msg_input)

    msg_input.submit(
        fn=agent_chat,
        inputs=[msg_input, chatbot, api_key_input, session_state, terminal_state, activity_state],
        outputs=[chatbot, terminal_output, activity_output]
    ).then(fn=lambda: "", inputs=None, outputs=msg_input)

demo.launch(theme=gr.themes.Monochrome(), debug=True, share=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://4b9e2653960982449b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
